# 01 Run Orchestrator

역할: runtime과 dataset 상태를 맞추고, baseline/category/manifest를 선택해 runner를 실행한다.
산출물은 `summary.json`과 `log.txt` handoff까지만 확인한다.


# Condition Shift Baseline Run Notebook

이 노트북에서 하는 일:

- Colab/서버에서 Git checkout과 dataset 준비 상태를 맞춘다.
- `PatchCore`, `UniVAD` runner를 `single` 또는 `sequential` 모드로 실행한다.
- 실행 결과로 남는 `summary.json`, `log.txt` 경로와 존재 여부를 handoff table로 확인한다.

중요한 순서:

1. Drive mount
2. Git checkout 또는 pull
3. checkout된 repo 기준으로 path/helper import
4. dataset/bootstrap/setup/run

분석과 figure export는 각각 `02_analysis_dashboard.ipynb`, `03_figure_export.ipynb`에서 수행한다.


## Cell Role: Drive Mount

Colab에서 Drive를 mount한다. 로컬 Jupyter에서는 `google.colab`이 없으므로 이 셀은 자동으로 skip된다.
Drive는 dataset tar, checkpoint backup, mask backup 같은 runtime 외부 자산 접근에만 사용한다.


In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is not available; skipping Drive mount for local runtime.')


Mounted at /content/drive


## Cell Role: Git Checkout

runner/helper를 import하기 전에 `/content/ReGraM`을 clone 또는 pull 한다.
이 순서를 지켜야 Colab 런타임에 남아 있던 예전 코드가 먼저 import되는 충돌을 피할 수 있다.


In [2]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/outSeop/ReGraM.git'
cwd = Path.cwd().resolve()
local_repo = next((p.resolve() for p in [cwd, *cwd.parents] if p.exists() and p.name == 'ReGraM'), None)

if Path('/content').exists():
    REPO_DIR = Path('/content/ReGraM')
    git_cmd = (
        f'if [ -d "{REPO_DIR}/.git" ]; then git -C "{REPO_DIR}" pull --ff-only; '
        f'else git clone "{REPO_URL}" "{REPO_DIR}"; fi'
    )
    print(git_cmd)
    subprocess.run(['bash', '-lc', git_cmd], check=True)
    CHECKOUT_ROOT = REPO_DIR.resolve()
else:
    CHECKOUT_ROOT = local_repo or cwd
    print('local runtime detected; skipping Colab clone/pull')

print('checkout root =', CHECKOUT_ROOT)


if [ -d "/content/ReGraM/.git" ]; then git -C "/content/ReGraM" pull --ff-only; else git clone "https://github.com/outSeop/ReGraM.git" "/content/ReGraM"; fi
checkout root = /content/ReGraM


## Cell Role: Path and Helper Setup

방금 checkout된 repo를 기준으로 `REPO_ROOT`, `EXP_ROOT`, `REPORT_ROOT`를 확정하고 Python helper를 import한다.
이 셀 이후의 모든 경로와 runner command는 여기서 확정된 root를 따른다.


In [3]:
from pathlib import Path
import importlib
import subprocess
import sys

import pandas as pd
import torch
from IPython.display import display

colab_checkout = Path('/content/ReGraM')
cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and p.name == 'ReGraM'), cwd)
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']
SRC_ROOT = EXP_ROOT / 'src'
CORE_SRC = SRC_ROOT / 'core'
UNIVAD_SRC = SRC_ROOT / 'univad'
for source_root in (SRC_ROOT, CORE_SRC, UNIVAD_SRC):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

# Colab/Jupyter keeps imported modules alive after git pull. Reload notebook helpers
# so rerunning this cell picks up the current checkout without a full runtime restart.
for module_name in (
    'orchestration.notebook_orchestration',
    'notebook_orchestration',
    'setup_runtime',
):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from orchestration.notebook_orchestration import (
    build_baseline_specs,
    build_requested_run_configs,
    discover_manifest_names,
    display_environment_summary,
    display_experiment_preset,
    display_run_plan,
    display_title,
    load_experiment_config,
    model_config_from_experiment_config,
    model_sweep_from_experiment_config,
    resolve_manifest_paths,
    resolve_requested_baselines,
    run_execution_queue,
    validate_baseline_names,
)
from setup_runtime import default_univad_setup_flags, evaluate_baseline_readiness, run_baseline_setup

DEFAULT_PATCHCORE_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
run_history = []

display_environment_summary(REPO_ROOT, EXP_ROOT, REPORT_ROOT)
print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Environment Summary

,key,value
0,REPO_ROOT,/content/ReGraM
1,EXP_ROOT,/content/ReGraM/experiments/validation/conditi...
2,REPORT_ROOT,/content/ReGraM/experiments/validation/conditi...


REPO_ROOT = /content/ReGraM
EXP_ROOT = /content/ReGraM/experiments/validation/condition_shift_baseline


## Cell Role: Raw Dataset Load

Drive에 있는 raw MVTec LOCO tar를 Colab runtime의 repo data 경로로 복사/해제한다.
이미 `data/row/mvtec_loco_anomaly_detection`이 있으면 재해제하지 않는다.


In [4]:
from pathlib import Path
import shutil
import subprocess

drive_tar = Path("/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz")
runtime_tar = Path("/content/mvtec_loco_anomaly_detection.tar.gz")
runtime_row = REPO_ROOT / "data" / "row"
runtime_root = runtime_row / "mvtec_loco_anomaly_detection"


def find_nested_dataset_root(base_dir: Path, dataset_name: str):
    direct = base_dir / dataset_name
    if direct.exists():
        return direct
    matches = [path for path in base_dir.rglob(dataset_name) if path.is_dir()]
    if not matches:
        return None
    matches = sorted(matches, key=lambda path: len(path.parts))
    return matches[0]


def normalize_dataset_root(base_dir: Path, dataset_name: str):
    target_root = base_dir / dataset_name
    discovered_root = find_nested_dataset_root(base_dir, dataset_name)
    if discovered_root is None:
        return target_root, False
    if discovered_root == target_root:
        return target_root, False
    print(f"normalize dataset root: {discovered_root} -> {target_root}")
    target_root.parent.mkdir(parents=True, exist_ok=True)
    if target_root.exists():
        raise RuntimeError(f"target dataset root already exists: {target_root}")
    shutil.move(str(discovered_root), str(target_root))
    return target_root, True


print("drive_tar exists:", drive_tar.exists())
print("runtime_root exists:", runtime_root.exists())

runtime_row.mkdir(parents=True, exist_ok=True)

if not runtime_root.exists():
    if not runtime_tar.exists():
        subprocess.run(["cp", str(drive_tar), str(runtime_tar)], check=True)
    subprocess.run(
        ["tar", "-xf", str(runtime_tar), "-C", str(runtime_row)],
        check=True,
    )

runtime_root, normalized = normalize_dataset_root(runtime_row, "mvtec_loco_anomaly_detection")
print("normalized:", normalized)
print("done")
print(runtime_root.exists(), runtime_root)


drive_tar exists: True
runtime_root exists: False
normalize dataset root: /content/ReGraM/data/row/content/mvtec_loco_anomaly_detection -> /content/ReGraM/data/row/mvtec_loco_anomaly_detection
normalized: True
done
True /content/ReGraM/data/row/mvtec_loco_anomaly_detection


## Cell Role: Optional Dataset Bootstrap

prepared dataset, small report artifact, 기타 보조 자산을 별도 bootstrap script로 확인한다.
현재는 `--dry-run`이므로 실제 복사 없이 무엇이 필요한지만 점검한다.


In [5]:
bootstrap_cmd = [
    sys.executable,
    str(EXP_ROOT / 'colab' / 'bootstrap_runtime.py'),
    '--dry-run',
]
print(' '.join(bootstrap_cmd))
subprocess.run(bootstrap_cmd, cwd=REPO_ROOT, check=True)


/usr/bin/python3 /content/ReGraM/experiments/validation/condition_shift_baseline/colab/bootstrap_runtime.py --dry-run


CompletedProcess(args=['/usr/bin/python3', '/content/ReGraM/experiments/validation/condition_shift_baseline/colab/bootstrap_runtime.py', '--dry-run'], returncode=0)

## Cell Role: Experiment Scope Config

이번 실행의 baseline, category, manifest 범위, wandb 설정, readiness 정책을 결정한다.
모델 하이퍼파라미터와 sweep은 `EXPERIMENT_CONFIG_PATH`가 가리키는 YAML의 `models` / `sweep` 섹션을 읽는다.
`CONFIGURED_MANIFEST_NAMES`가 비어 있지 않으면 그 목록만 실행하고, 전체 auto-discovery는 쓰지 않는다.
`SELECTED_SEVERITIES`가 비어 있으면 low/medium/high 전체를 실행하고, 값이 있으면 해당 severity만 실행한다.


In [7]:
RUN_MODE = 'sequential'  # choose 'single' or 'sequential'
RUN_BASELINE = ['UniVAD', 'PatchCore']      # used only when RUN_MODE == 'single'
SEQUENTIAL_BASELINES = ['UniVAD', 'PatchCore']
DASHBOARD_BASELINES = ['UniVAD', 'PatchCore']
# breakfast_box, juice_bottle, screw_bag, pushpins, splicing_connectors
CATEGORIES = ['breakfast_box']
AUTO_DISCOVER_MANIFESTS = False
CONFIGURED_MANIFEST_NAMES = ['query_gaussian_noise.jsonl', 'query_gaussian_blur.jsonl', 'query_position_shift.jsonl']
SELECTED_SEVERITIES = ['high']  # [] means all; examples: ['low'], ['medium'], ['high'], ['low', 'high']
EXCLUDED_MANIFEST_NAMES = {'query_identity.jsonl', 'query_multi.jsonl'}
WANDB_PROJECT = 'regram-condition-shift-2'
WANDB_MODE = 'online'
SHOW_RUNNER_OUTPUT = True
RUNNER_OUTPUT_TAIL_LINES = 40
RUN_ONLY_READY_BASELINES = True
STOP_ON_FAILURE = True
TOP_K_WORST_SHIFTS = 10
EXPERIMENT_CONFIG_PATH = EXP_ROOT / 'configs' / 'experiment_template.yaml'

try:
    load_experiment_config
except NameError:
    import orchestration.notebook_orchestration as notebook_orchestration_module
    importlib.reload(notebook_orchestration_module)
    from orchestration.notebook_orchestration import (
        load_experiment_config,
        model_config_from_experiment_config,
        model_sweep_from_experiment_config,
    )

EXPERIMENT_CONFIG = load_experiment_config(EXPERIMENT_CONFIG_PATH)
MODEL_CONFIG = model_config_from_experiment_config(EXPERIMENT_CONFIG)
MODEL_SWEEP = model_sweep_from_experiment_config(EXPERIMENT_CONFIG)

RAW_LOCO_ROOT = REPO_ROOT / 'data' / 'row' / 'mvtec_loco_anomaly_detection'
UNIVAD_CAPTION_DATA_ROOT = REPO_ROOT / 'data' / 'mvtec_loco_caption'
UNIVAD_SETUP_FLAGS = default_univad_setup_flags()

BASELINE_SPECS = build_baseline_specs(
    repo_root=REPO_ROOT,
    exp_root=EXP_ROOT,
    raw_loco_root=RAW_LOCO_ROOT,
    univad_caption_data_root=UNIVAD_CAPTION_DATA_ROOT,
    default_patchcore_device=DEFAULT_PATCHCORE_DEVICE,
    model_config=MODEL_CONFIG,
    model_sweep=MODEL_SWEEP,
)
validate_baseline_names(DASHBOARD_BASELINES, specs=BASELINE_SPECS, label='dashboard')
REQUESTED_BASELINES = resolve_requested_baselines(
    RUN_MODE,
    RUN_BASELINE,
    SEQUENTIAL_BASELINES,
    specs=BASELINE_SPECS,
)
ACTIVE_MANIFEST_NAMES = discover_manifest_names(
    manifest_roots=MANIFEST_ROOT_CANDIDATES,
    configured_names=CONFIGURED_MANIFEST_NAMES,
    auto_discover=AUTO_DISCOVER_MANIFESTS,
    excluded_names=EXCLUDED_MANIFEST_NAMES,
)
manifest_paths = resolve_manifest_paths(ACTIVE_MANIFEST_NAMES, MANIFEST_ROOT_CANDIDATES)
requested_run_configs = build_requested_run_configs(
    active_baselines=REQUESTED_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    manifest_paths=manifest_paths,
    manifest_names=ACTIVE_MANIFEST_NAMES,
    report_root=REPORT_ROOT,
    wandb_project=WANDB_PROJECT,
    wandb_mode=WANDB_MODE,
    selected_severities=SELECTED_SEVERITIES,
)
dashboard_run_configs = build_requested_run_configs(
    active_baselines=DASHBOARD_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    manifest_paths=manifest_paths,
    manifest_names=ACTIVE_MANIFEST_NAMES,
    report_root=REPORT_ROOT,
    wandb_project=WANDB_PROJECT,
    wandb_mode=WANDB_MODE,
    selected_severities=SELECTED_SEVERITIES,
)
baseline_status_df = evaluate_baseline_readiness(REQUESTED_BASELINES, specs=BASELINE_SPECS, categories=CATEGORIES)
run_configs = list(requested_run_configs)
skipped_run_configs = []

display_experiment_preset(
    run_mode=RUN_MODE,
    requested_baselines=REQUESTED_BASELINES,
    dashboard_baselines=DASHBOARD_BASELINES,
    categories=CATEGORIES,
    manifest_names=ACTIVE_MANIFEST_NAMES,
    wandb_mode=WANDB_MODE,
    stop_on_failure=STOP_ON_FAILURE,
    univad_flags=UNIVAD_SETUP_FLAGS,
    specs=BASELINE_SPECS,
    requested_run_configs=requested_run_configs,
)


## Experiment Preset

RUN_MODE=`sequential` | requested baselines=`UniVAD, PatchCore` | dashboard baselines=`UniVAD, PatchCore`

,run_mode,requested_baselines,dashboard_baselines,category_count,manifest_count,selected_severities,wandb_mode,stop_on_failure,download_missing_univad_checkpoints,auto_prepare_univad_caption_dataset,...,auto_fix_univad_transformers_stack,auto_fix_univad_runtime_deps,restore_model_checkpoints_from_drive,backup_model_checkpoints_to_drive,auto_mount_google_drive_for_model_checkpoints,model_checkpoint_drive_backup_root,restore_univad_grounding_masks_from_drive,backup_univad_grounding_masks_to_drive,auto_mount_google_drive_for_univad_masks,univad_mask_drive_backup_root
0,sequential,"UniVAD, PatchCore","UniVAD, PatchCore",1,3,high,online,True,True,True,...,True,True,True,True,True,/content/drive/MyDrive/ReGraM/model_checkpoints,True,True,True,/content/drive/MyDrive/ReGraM/univad_masks/mvt...


,baseline,runner_name,data_root,device,model_config,wandb_group,wandb_mode,runner_outputs,notes
0,UniVAD,UniVAD manifest shift evaluation,/content/ReGraM/data/mvtec_loco_caption,cuda,"image_size=224, k_shot=1, round=0, export_heat...",univad,online,single summary json per category under reports...,"UniVAD needs CUDA, recursive clone/submodules,..."
1,PatchCore,PatchCore manifest shift evaluation,/content/ReGraM/data/row/mvtec_loco_anomaly_de...,cuda,"resize=256, imagesize=224, batch_size=1, num_w...",patchcore,online,single summary json per category under reports...,PatchCore runner consumes raw LOCO directly an...


,manifest_name
0,query_gaussian_noise.jsonl
1,query_gaussian_blur.jsonl
2,query_position_shift.jsonl


,order,baseline,display_baseline,model_variant,category,manifest_count,severities,summary_name,device,model_config,wandb
0,1,UniVAD,UniVAD,default,breakfast_box,3,high,breakfast_box_multi_high.json,cuda,"image_size=224, k_shot=1, round=0, export_heat...",on
1,2,PatchCore,PatchCore,default,breakfast_box,3,high,breakfast_box_multi_high.json,cuda,"resize=256, imagesize=224, batch_size=1, num_w...",on


## Cell Role: Run Matrix Preview

실제로 실행될 baseline/category/manifest 수와 runner command를 실행 전에 확인한다.
여기서 manifest count가 예상과 다르면 아래 실행 셀로 내려가지 않는다.


In [8]:
display_title('Current Run Matrix', 'Single mode executes one baseline. Sequential mode executes the listed baselines in order and keeps a shared comparison dashboard.')
command_preview_df = pd.DataFrame([
    {
        'baseline': config['baseline'],
        'category': config['category'],
        'manifest_count': len(config['manifest_paths']),
        'severities': ', '.join(config.get('selected_severities', [])) or 'all',
        'device': config['device'],
        'summary_name': config['summary_path'].name,
        'log_name': config['log_path'].name,
        'runner_cmd': ' '.join(config['runner_cmd']),
    }
    for config in requested_run_configs
])
display(command_preview_df)


## Current Run Matrix

Single mode executes one baseline. Sequential mode executes the listed baselines in order and keeps a shared comparison dashboard.

,baseline,category,manifest_count,severities,device,summary_name,log_name,runner_cmd
0,UniVAD,breakfast_box,3,high,cuda,breakfast_box_multi_high.json,breakfast_box_multi_high.log.txt,/usr/bin/python3 /content/ReGraM/experiments/v...
1,PatchCore,breakfast_box,3,high,cuda,breakfast_box_multi_high.json,breakfast_box_multi_high.log.txt,/usr/bin/python3 /content/ReGraM/experiments/v...


## Cell Role: Baseline Setup and Readiness

외부 repo, dependency, checkpoint, dataset, grounding mask 상태를 baseline별로 점검하고 필요한 setup을 수행한다.
`RUN_ONLY_READY_BASELINES`가 켜져 있으면 준비되지 않은 run은 execution queue에서 제외한다.


In [9]:
display_title(
    'Baseline Setup & Readiness',
    'Setup/check external repos, runtime dependencies, checkpoints, datasets, and masks before execution.',
)
setup_result = run_baseline_setup(
    requested_baselines=REQUESTED_BASELINES,
    specs=BASELINE_SPECS,
    categories=CATEGORIES,
    raw_loco_root=RAW_LOCO_ROOT,
    settings=UNIVAD_SETUP_FLAGS,
    requested_run_configs=requested_run_configs,
    run_only_ready_baselines=RUN_ONLY_READY_BASELINES,
)
setup_df = setup_result['setup_df']
baseline_status_df = setup_result['baseline_status_df']
run_configs = setup_result['run_configs']
skipped_run_configs = setup_result['skipped_run_configs']

with pd.option_context('display.max_colwidth', None, 'display.max_rows', 80):
    display(setup_df)
if 'UniVAD' in REQUESTED_BASELINES:
    for title, key in [
        ('UniVAD Dataset Status', 'univad_dataset_status_df'),
        ('UniVAD Local Dependency Status', 'univad_local_dependency_status_df'),
        ('UniVAD Checkpoint Status', 'univad_checkpoint_status_df'),
        ('UniVAD Grounding Mask Status', 'univad_mask_status_df'),
        ('UniVAD Missing Mask Preview', 'univad_missing_mask_preview_df'),
    ]:
        display_title(title)
        with pd.option_context('display.max_colwidth', None, 'display.max_rows', 80):
            display(setup_result[key])

display_title('Baseline Readiness')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', 80):
    display(baseline_status_df)
if skipped_run_configs:
    display_title('Skipped Runs')
    status_by_baseline = baseline_status_df.set_index('baseline').to_dict('index') if not baseline_status_df.empty else {}
    skipped_df = pd.DataFrame([
        {
            'baseline': config['baseline'],
            'category': config['category'],
            'blockers': status_by_baseline.get(config['baseline'], {}).get('blockers', '-'),
            'missing_mask_paths': status_by_baseline.get(config['baseline'], {}).get('missing_mask_paths', '-'),
            'summary_path': str(config['summary_path']),
            'log_path': str(config['log_path']),
        }
        for config in skipped_run_configs
    ])
    with pd.option_context('display.max_colwidth', None, 'display.max_rows', 80):
        display(skipped_df)

display_title('Execution Queue', f'Prepared `{len(run_configs)}` runnable jobs.')
display_run_plan(run_configs)


## Baseline Setup & Readiness

Setup/check external repos, runtime dependencies, checkpoints, datasets, and masks before execution.

patched UniVAD optional pydensecrf fallback: /content/ReGraM/external/UniVAD/utils/crf.py
patched UniVAD re-seg loop guard: /content/ReGraM/external/UniVAD/UniVAD.py
prepare UniVAD caption dataset: /content/ReGraM/data/mvtec_loco_caption <- /content/ReGraM/data/row/mvtec_loco_anomaly_detection
model checkpoint state: file=sam_hq_vit_h.pth session=missing drive=ready action=restore_from_drive
model checkpoint state: file=groundingdino_swint_ogc.pth session=missing drive=ready action=restore_from_drive
model checkpoint download: action=skip reason=session_ready
model checkpoint state: file=sam_hq_vit_h.pth session=ready drive=ready action=keep_drive_cache
model checkpoint state: file=groundingdino_swint_ogc.pth session=ready drive=ready action=keep_drive_cache
install compatible UniVAD runtime dependency stack
run pip: uninstall -y torchao torchao-nightly
run pip: uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
run pip: install --upg

,baseline,external_dir,data_root,mask_root,groundingdino_dir,checkpoint_root,groundingdino_importable,missing_local_paths,caption_dataset_prepared,masks_generated,...,mask_restore_categories,mask_restore_file_count,mask_backup_status,mask_backup_path,mask_backup_categories,mask_backup_file_count,missing_data_paths,missing_mask_paths,setup_status,note
0,UniVAD,/content/ReGraM/external/UniVAD,/content/ReGraM/data/mvtec_loco_caption,/content/ReGraM/external/UniVAD/masks/mvtec_loco_caption,/content/ReGraM/external/UniVAD/models/GroundingDINO,/content/ReGraM/external/UniVAD/pretrained_ckpts,False,-,True,False,...,breakfast_box,626.0,ok,/content/drive/MyDrive/ReGraM/univad_masks/mvtec_loco_caption,breakfast_box,626.0,-,-,restart_required,installed/updated UniVAD dependency stack(s); restart runtime once and rerun notebook from the top. Details: runtime_dependency_unavailable: installed compatible UniVAD runtime deps with headless OpenCV; restart runtime and rerun notebook from the top | torch_stack_unavailable: installed compatible torch stack; restart runtime and rerun notebook from the top | transformers_stack_unavailable: installed compatible transformers stack; restart runtime and rerun notebook from the top
1,PatchCore,/content/ReGraM/external/patchcore-inspection.clean,/content/ReGraM/data/row/mvtec_loco_anomaly_detection,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ready,Repo synced and lightweight runtime deps installed.


## UniVAD Dataset Status

,baseline,required_data_path,exists,path
0,UniVAD,/content/ReGraM/data/mvtec_loco_caption/meta.json,True,/content/ReGraM/data/mvtec_loco_caption/meta.json
1,UniVAD,/content/ReGraM/data/mvtec_loco_caption/breakfast_box,True,/content/ReGraM/data/mvtec_loco_caption/breakfast_box


...

""


## Baseline Readiness

,baseline,requested,device,data_root,external_repo_exists,cuda_required,cuda_available,checkpoint_required,checkpoint_root,checkpoint_ready,...,missing_data_paths,mask_root,masks_ready,missing_mask_paths,required_python_modules,python_modules_ready,ready,blockers,notes,will_run
0,UniVAD,True,cuda,/content/ReGraM/data/mvtec_loco_caption,True,True,True,True,/content/ReGraM/external/UniVAD/pretrained_ckpts,True,...,-,/content/ReGraM/external/UniVAD/masks/mvtec_loco_caption,True,-,"groundingdino, torchmetrics",True,True,ready,"UniVAD needs CUDA, recursive clone/submodules, MVTec-Caption style LOCO data, precomputed grounding masks, editable GroundingDINO, and local checkpoints.",True
1,PatchCore,True,cuda,/content/ReGraM/data/row/mvtec_loco_anomaly_detection,True,False,True,False,,True,...,-,,True,-,-,True,True,ready,PatchCore runner consumes raw LOCO directly and falls back to CPU automatically.,True


## Execution Queue

Prepared `2` runnable jobs.

,order,baseline,display_baseline,model_variant,category,manifest_count,severities,summary_name,device,model_config,wandb
0,1,UniVAD,UniVAD,default,breakfast_box,3,high,breakfast_box_multi_high.json,cuda,"image_size=224, k_shot=1, round=0, export_heat...",on
1,2,PatchCore,PatchCore,default,breakfast_box,3,high,breakfast_box_multi_high.json,cuda,"resize=256, imagesize=224, batch_size=1, num_w...",on


## Cell Role: Execute Run Queue

준비가 끝난 run queue를 순서대로 실행한다.
이 셀은 오래 걸릴 수 있고, runner output은 `SHOW_RUNNER_OUTPUT` / `RUNNER_OUTPUT_TAIL_LINES` 설정을 따른다.


In [9]:
run_history = run_execution_queue(
    run_configs=run_configs,
    repo_root=REPO_ROOT,
    show_runner_output=SHOW_RUNNER_OUTPUT,
    runner_output_tail_lines=RUNNER_OUTPUT_TAIL_LINES,
    stop_on_failure=STOP_ON_FAILURE,
)


## Execution Log

Started at `2026-05-06 03:24:35 UTC` with `2` scheduled runs.

[1/2] START UniVAD / breakfast_box @ 2026-05-06 03:24:35 UTC
summary_path = /content/ReGraM/experiments/validation/condition_shift_baseline/reports/univad_manifest_shift/breakfast_box_multi_high.json
log_path = /content/ReGraM/experiments/validation/condition_shift_baseline/reports/univad_manifest_shift/logs/breakfast_box_multi_high.log.txt
command = /usr/bin/python3 /content/ReGraM/experiments/validation/condition_shift_baseline/src/univad/run_manifest_shift.py --category breakfast_box --manifest /content/ReGraM/manifests/query_gaussian_noise.jsonl /content/ReGraM/manifests/query_gaussian_blur.jsonl /content/ReGraM/manifests/query_position_shift.jsonl --data-root /content/ReGraM/data/mvtec_loco_caption --device cuda --output /content/ReGraM/experiments/validation/condition_shift_baseline/reports/univad_manifest_shift --log-dir /content/ReGraM/experiments/validation/condition_shift_baseline/reports/univad_manifest_shift/logs --image-size 224 --k-shot 1 --round 0 --export-heatmaps --hea

## Execution Summary

Finished at `2026-05-06 03:48:53 UTC`.

,order,baseline,display_baseline,model_variant,category,status,returncode,elapsed_sec,started_at,finished_at,summary_exists,log_exists,summary_path,log_path
0,1,UniVAD,UniVAD,default,breakfast_box,success,0,1250.85,2026-05-06 03:24:35 UTC,2026-05-06 03:45:26 UTC,True,True,/content/ReGraM/experiments/validation/conditi...,/content/ReGraM/experiments/validation/conditi...
1,2,PatchCore,PatchCore,default,breakfast_box,success,0,207.31,2026-05-06 03:45:26 UTC,2026-05-06 03:48:53 UTC,True,True,/content/ReGraM/experiments/validation/conditi...,/content/ReGraM/experiments/validation/conditi...


## Cell Role: Output Handoff

runner가 남긴 `summary.json`과 `log.txt`의 존재 여부와 경로만 확인한다.
결과 해석, 비교 테이블, 시각화는 `02_analysis_dashboard.ipynb`에서 수행한다.


In [10]:
summary_sources = dashboard_run_configs if 'dashboard_run_configs' in globals() else requested_run_configs
handoff_rows = []
for config in summary_sources:
    summary_path = config['summary_path']
    log_path = config['log_path']
    handoff_rows.append(
        {
            'baseline': config['baseline'],
            'category': config['category'],
            'summary_exists': summary_path.exists(),
            'summary_path': str(summary_path),
            'log_exists': log_path.exists(),
            'log_path': str(log_path),
        }
    )

display_title(
    'Output Handoff',
    'Open notebook/02_analysis_dashboard.ipynb for tables, plots, and interpretation.',
)
display(pd.DataFrame(handoff_rows))


## Output Handoff

Open notebook/02_analysis_dashboard.ipynb for tables, plots, and interpretation.

,baseline,category,summary_exists,summary_path,log_exists,log_path
0,UniVAD,breakfast_box,True,/content/ReGraM/experiments/validation/conditi...,True,/content/ReGraM/experiments/validation/conditi...
1,PatchCore,breakfast_box,True,/content/ReGraM/experiments/validation/conditi...,True,/content/ReGraM/experiments/validation/conditi...
